# Bistable neuron -- model overview

Two things, in order:

1. **Persistent memory** -- a two-pulse test on a single neuron: a short
   negative pulse turns it *off*, a short positive pulse turns it back *on*,
   and it should stay in whichever state it's in through the silence between
   pulses. This is the property the MNIST network (notebook 2) relies on to
   hold information across its `T_wait` silent steps.
2. **Phase-plane portraits** -- the (v, w) vector field that explains *why*
   the pulse test behaves the way it does: nullclines, fixed points
   (stable / unstable / saddle), and which initial conditions end up at rest
   vs. spiking.

Both use the exact same `bistable_neuron` function from `neuron_model.py` --
nothing here is redefined, so what you see is exactly what the network in
notebook 2 runs internally.

In [ ]:
from params import *
import numpy as np
import matplotlib.pyplot as plt
import torch

import neuron_model as neuron
import plotting_helpers as plot_utils

print("a, eps, c, d, v_th :", a, eps, c, d, v_th)
print("w0 (default)       :", neuron.w0)

## Part 1 -- Persistent memory: two-pulse switching test

### `dt`, `steps`, and "timestep" -- what each one actually is

Three different things, easy to conflate:

- **`dt`** -- the RK4 integrator's nominal step size. A number in the ODE's
  own abstract units, not ms or seconds.
- **`steps`** -- how many `dt`-sized increments get folded into *one*
  `bistable_neuron()` call: `dt_eff = dt * steps`. With the defaults
  (`dt=0.01`, `steps=5`), one call integrates `dt_eff = 0.05` time-units.
- **"Timestep" (network sense)** -- one call to `bistable_neuron()`. This is
  the unit `T` / `T_wait` / `T_eval` are counted in, in the MNIST network
  (notebook 2). Each one is worth `dt_eff` time-units of actual ODE dynamics.

Everything below -- pulse widths, silence, plot x-axis -- is specified in
**network steps**, so it lines up 1:1 with `T` / `T_wait` in the network.

In [ ]:
# -- pulse protocol, in NETWORK STEPS (same units as T / T_wait) --
w0_val           = -0.5   # override neuron.w0 for this experiment; try flipping the sign
t_silence_steps  = 500    # silence before the first pulse and after the last
t_start_steps    = 500    # step index where the first pulse begins
t_pulse_steps    = 100    # pulse width, in steps
t_between_steps  = 1000   # silence between the two pulses
I_amp            = 1.0    # pulse amplitude

steps  = 5                 # substeps folded into each bistable_neuron() call
dt_eff = dt * steps         # actual model-time advanced per call

n_steps  = t_start_steps + t_pulse_steps + t_between_steps + t_pulse_steps + t_silence_steps
step_idx = np.arange(n_steps)

def build_pulse_input():
    # Biphasic pulse pair: - then +, with silence before/after/between.
    I_ext = np.zeros(n_steps)
    p1_on, p1_off = t_start_steps, t_start_steps + t_pulse_steps
    p2_on = p1_off + t_between_steps
    p2_off = p2_on + t_pulse_steps

    I_ext[p1_on:p1_off] = -I_amp
    I_ext[p2_on:p2_off] = I_amp

    print(f"  Pulse 1 (-) : steps {p1_on}-{p1_off}  ({p1_on*dt_eff:.2f}-{p1_off*dt_eff:.2f} model-time)")
    print(f"  Pulse 2 (+) : steps {p2_on}-{p2_off}  ({p2_on*dt_eff:.2f}-{p2_off*dt_eff:.2f} model-time)")
    return I_ext

I_ext = build_pulse_input()

### Sanity check: pulse width vs. the neuron's own relaxation timescale

A pulse much shorter than `1/eps` (the recovery variable's own timescale)
won't reliably switch the neuron; a pulse much longer will drive it through
many spike cycles before it can turn off. Rule of thumb: aim for a ratio
somewhere around 1-5.

In [ ]:
print("dt_eff per call             :", dt_eff)
print("Pulse duration (model-time)  :", t_pulse_steps * dt_eff)
print("Silence duration (model-time):", t_between_steps * dt_eff)
print("Relaxation timescale ~1/eps  :", 1 / eps)
print("Pulse / relaxation ratio     :", (t_pulse_steps * dt_eff) / (1 / eps))

In [ ]:
v, w = torch.tensor(0.0), torch.tensor(0.0)
v_traces, w_traces, spike_flags = [], [], []

for step in range(n_steps):
    I_t = torch.tensor(I_ext[step], dtype=torch.float32)
    v, w, spiked = neuron.bistable_neuron(v, w, I_t, dt=dt, steps=steps, w0=w0_val)
    v_traces.append(v.item())
    w_traces.append(w.item())
    spike_flags.append(spiked.item())

spike_count = int(np.sum(np.array(spike_flags) > 0.5))
print("total spikes:", spike_count)

In [ ]:
plot_utils.plot_voltage_traces(
    step_idx, I_ext, v_traces, w_traces, spike_flags, dt_eff,
    title=rf"Persistent memory for $w_0={w0_val}$,  $I_{{pulse}}=\pm{I_amp}$,  spikes={spike_count}",
)

## Part 2 -- Phase-plane portraits (2D, z = 0)

Same `w0` as the pulse test above, three panels: `I_ext = 0`, `+I_ext`,
`-I_ext`. Solid black curve is the v-nullcline, dashed orange is the
w-nullcline; fixed points are marked stable (filled), unstable (open), or
saddle (cross); trajectories are colour-coded by where they end up.

Compare this to Part 1: the pulse test's negative phase should correspond to
a panel here where the reset point (orange square) sits in the *rest*
basin, and the positive phase to a panel where it sits in the *spiking*
basin -- that's the whole mechanism behind persistent memory in this model.
See the project's bifurcation-analysis notes for the full derivation of
`I_TC` and the separatrix.

In [ ]:
from phase_plane_helpers import plot_input_sweep

_ = plot_input_sweep(w0=w0_val, I_ext=I_amp)

## Part 3 -- Third (slow) timescale, z -- TODO

Everything above is the 2-timescale model (fast `v`, slow `w`), i.e. the
`z = 0` slice. A 3-timescale extension -- an even slower `z` on top of
`(v, w)` -- exists in some of Daria's other analyses but wasn't part of the
files handed off into this package, so it isn't implemented here yet.

`phase_plane_helpers.plot_phase_plane_with_z(...)` is a stub that raises
`NotImplementedError` with pointers on how to fill it in (see its
docstring), or ask Daria for the original script with the "ultraslow-z
survey machinery" if you need the exact equations. Left commented out below
so running the whole notebook doesn't error.

In [ ]:
# from phase_plane_helpers import plot_phase_plane_with_z
# plot_phase_plane_with_z(w0=w0_val, I_ext=I_amp)   # not implemented yet -- see docstring
print("z-timescale phase portrait: not yet implemented -- see the markdown cell above.")